# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rish7186/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector is built by selecting the input features, handling missing values, and separating the target variable. The `total_bedrooms` column contains missing values, so these are filled using the median. The dataset contains only numerical features, so no categorical encoding is required. The target variable (`median_house_value`) is excluded from the feature vector to prevent data leakage.

In [2]:
import duckdb
import pandas as pd

# Placeholder for loading the housing dataset into df
# For example:
# df = pd.read_csv("path/to/your/housing.csv")
# Or create a dummy DataFrame for demonstration purposes to avoid the NameError:
data = {
    'longitude': [-122.23, -122.22],
    'latitude': [37.88, 37.86],
    'housing_median_age': [41.0, 21.0],
    'total_rooms': [880.0, 7099.0],
    'total_bedrooms': [129.0, 1106.0],
    'population': [322.0, 2401.0],
    'households': [126.0, 1138.0],
    'median_income': [8.3252, 8.3014],
    'median_house_value': [452600.0, 358500.0]
}
df = pd.DataFrame(data)


# Create DuckDB connection
con = duckdb.connect()

# Register dataframe
con.register("housing", df)

# Build feature vector using SQL
features = con.sql("""
SELECT
    longitude,
    latitude,
    housing_median_age,
    total_rooms,
    COALESCE(total_bedrooms,
             (SELECT MEDIAN(total_bedrooms) FROM housing)) AS total_bedrooms,
    population,
    households,
    median_income
FROM housing
""").df()

# Target variable
target = con.sql("""
SELECT median_house_value
FROM housing
""").df()["median_house_value"]

print("Feature Vector Shape:", features.shape)
print("Target Shape:", target.shape)

# Display first few rows
display(features.head())

Feature Vector Shape: (2, 8)
Target Shape: (2,)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature            | Meaning                                      | Missing Values                            | Categorical? | Available Before Prediction? |
| ------------------ | -------------------------------------------- | ----------------------------------------- | ------------ | ---------------------------- |
| longitude          | Geographic longitude of the housing district | No                                        | No           | Yes                          |
| latitude           | Geographic latitude of the housing district  | No                                        | No           | Yes                          |
| housing_median_age | Median age of houses in the district         | No                                        | No           | Yes                          |
| total_rooms        | Total number of rooms in the district        | No                                        | No           | Yes                          |
| total_bedrooms     | Total number of bedrooms in the district     | Missing values are filled with the median | No           | Yes                          |
| population         | Total population in the district             | No                                        | No           | Yes                          |
| households         | Number of households in the district         | No                                        | No           | Yes                          |
| median_income      | Median income of households in the district  | No                                        | No           | Yes                          |


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display feature information

feature_info = {
    "longitude": "Location (Longitude)",
    "latitude": "Location (Latitude)",
    "housing_median_age": "Median age of houses",
    "total_rooms": "Total rooms",
    "total_bedrooms": "Total bedrooms (missing values filled with median)",
    "population": "District population",
    "households": "Number of households",
    "median_income": "Median household income"
}

print("Feature Summary:\n")

for feature, description in feature_info.items():
    missing = df[feature].isnull().sum()
    dtype = df[feature].dtype

    print(f"{feature}")
    print(f"  Meaning      : {description}")
    print(f"  Missing      : {missing}")
    print(f"  Data Type    : {dtype}")
    print(f"  Categorical? : No")
    print(f"  Available Before Prediction? : Yes\n")

Feature Summary:

longitude
  Meaning      : Location (Longitude)
  Missing      : 0
  Data Type    : float64
  Categorical? : No
  Available Before Prediction? : Yes

latitude
  Meaning      : Location (Latitude)
  Missing      : 0
  Data Type    : float64
  Categorical? : No
  Available Before Prediction? : Yes

housing_median_age
  Meaning      : Median age of houses
  Missing      : 0
  Data Type    : float64
  Categorical? : No
  Available Before Prediction? : Yes

total_rooms
  Meaning      : Total rooms
  Missing      : 0
  Data Type    : float64
  Categorical? : No
  Available Before Prediction? : Yes

total_bedrooms
  Meaning      : Total bedrooms (missing values filled with median)
  Missing      : 0
  Data Type    : float64
  Categorical? : No
  Available Before Prediction? : Yes

population
  Meaning      : District population
  Missing      : 0
  Data Type    : float64
  Categorical? : No
  Available Before Prediction? : Yes

households
  Meaning      : Number of household

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the selected features for data leakage. None of the input features are derived from the target variable (`median_house_value`), and there are no future-looking variables, timestamps, or product-decision flags in the dataset. All features represent information that would be available before predicting the house value. Therefore, the feature set does not contain obvious sources of data leakage.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# Create DuckDB connection
con = duckdb.connect()

# Register dataframe
con.register("housing", df)

# Show all available columns
print("Columns in the dataset:")
print(con.sql("DESCRIBE housing").df())

# Verify the target column is not included in the feature vector
print("\nTarget column:")
print("median_house_value")

print("\nFeature columns:")
print(features.columns.tolist())

if "median_house_value" in features.columns:
    print("\n Data Leakage Detected: Target is included as a feature!")
else:
    print("\n No data leakage detected: Target is not part of the feature vector.")

# Check for date/time columns (future information)
date_cols = [col for col in df.columns if "date" in col.lower() or "time" in col.lower()]

if date_cols:
    print("\n Date/Time columns found:", date_cols)
else:
    print("\n No date or timestamp columns found.")

Columns in the dataset:
          column_name column_type null   key default extra
0           longitude      DOUBLE  YES  None    None  None
1            latitude      DOUBLE  YES  None    None  None
2  housing_median_age      DOUBLE  YES  None    None  None
3         total_rooms      DOUBLE  YES  None    None  None
4      total_bedrooms      DOUBLE  YES  None    None  None
5          population      DOUBLE  YES  None    None  None
6          households      DOUBLE  YES  None    None  None
7       median_income      DOUBLE  YES  None    None  None
8  median_house_value      DOUBLE  YES  None    None  None

Target column:
median_house_value

Feature columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']

 No data leakage detected: Target is not part of the feature vector.

 No date or timestamp columns found.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded Fields:

1. median_house_value
   - Reason: This is the target (label) that the model is trying to predict. Including it as a feature would cause data leakage and produce unrealistically high model performance.

No other columns were excluded because all remaining variables are available before prediction and provide useful information for estimating house values.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Excluded fields and reasons

excluded_fields = {
    "median_house_value": "Target variable (label). Excluded from features to prevent data leakage."
}

print("Excluded Fields:\n")

for field, reason in excluded_fields.items():
    print(f"{field}")
    print(f"Reason: {reason}\n")

# Verify that the excluded field is not in the feature vector
if "median_house_value" in features.columns:
    print(" Error: Target column is present in the feature vector.")
else:
    print(" Verified: Target column is excluded from the feature vector.")

Excluded Fields:

median_house_value
Reason: Target variable (label). Excluded from features to prevent data leakage.

 Verified: Target column is excluded from the feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.